# Office-10 sensitivity: 16 runs

α=0.1; seeds 42 and 123. Five graph strengths plus three non-default reliability values. The default is trained once per seed. Change DATA_ROOT below to reuse prepared Office-10 data.

In [ ]:
from pathlib import Path
import os, sys, subprocess
PROJECT = Path.cwd().resolve()  # Set explicitly if the notebook kernel starts elsewhere.
required = ['run_sensitivity.py','summarize_sensitivity.py','icde/config.py','icde/__init__.py','reference/run_office10.py']
missing = [name for name in required if not (PROJECT/name).is_file()]
assert not missing, f"Incomplete extraction at {PROJECT}: {missing}. Run repair_sensitivity_sources.py."
DATA_ROOT = PROJECT / 'data'  # Change to the existing directory containing office10.
OUTPUT = PROJECT / 'outputs' / 'office10_sensitivity'
DEVICE = 'cuda'
WORKERS = 4
BATCH_SIZE = 32
AMP = True
RESUME = True
ENV = dict(os.environ)
ENV['PYTHONPATH'] = str(PROJECT) + os.pathsep + ENV.get('PYTHONPATH','')
args = [sys.executable, str(PROJECT/'run_sensitivity.py'), '--data-root', str(DATA_ROOT), '--output', str(OUTPUT), '--device', DEVICE, '--workers', str(WORKERS), '--batch-size', str(BATCH_SIZE)]
if AMP: args.append('--amp')
if RESUME: args.append('--resume')


In [ ]:
subprocess.run(args + ['--plan-only'], cwd=PROJECT, env=ENV, check=True)
import json
plan = json.loads((OUTPUT/'plan.json').read_text())
for r in plan: print(r['seed'], r['graph_lambda'], r['tempering_beta'])

In [ ]:
assert (DATA_ROOT/'office10').is_dir(), 'DATA_ROOT must contain office10'
subprocess.run(args, cwd=PROJECT, env=ENV, check=True)

In [ ]:
from IPython.display import display, Image
if (OUTPUT/'sensitivity_summary.csv').exists():
    print((OUTPUT/'sensitivity_summary.csv').read_text())
    display(Image(filename=str(OUTPUT/'sensitivity.png')))
else:
    print('Full plots require all 16 runs. Partial results are in sensitivity_per_run.csv.')